In [1]:
using ITensors
using ITensorMPS: op
using KrylovKit: schursolve, Arnoldi
using LinearAlgebra

In [4]:
struct iMPS2
    Γa::ITensor
    λa::ITensor
    Γb::ITensor
    λb::ITensor
    iMPS2(Γa::ITensor, λa::ITensor, Γb::ITensor, λb::ITensor) = new(Γa, λa, Γb, λb)
end

function iMPS2(Γa::Array{ComplexF64, 3}, λa::Array{ComplexF64, 2}, Γb::Array{ComplexF64, 3}, λb::Array{ComplexF64, 2})
    l02 = Index(size(λb, 2), tags="Link,n=0,l=2")
    l11 = Index(size(λa, 1), tags="Link,n=1,l=1")
    l12 = Index(size(λa, 2), tags="Link,n=1,l=2")
    l21 = Index(size(λb, 1), tags="Link,n=2,l=1")
    l22 = Index(size(λb, 2), tags="Link,n=2,l=2")

    s1 = Index(size(Γa, 2), tags="Site,n=1")
    s2 = Index(size(Γb, 2), tags="Site,n=2")

    Γa_itensor = ITensor(Γa, (l02, s1, l11))
    λa_itensor = ITensor(λa, (l11, l12))
    Γb_itensor = ITensor(Γb, (l12, s2, l21))
    λb_itensor = ITensor(λb, (l21, l22))
    return iMPS2(Γa_itensor, λa_itensor, Γb_itensor, λb_itensor)
end

function reset_indices(ψ::iMPS2)
    l02, l11, l12, l21, l22 = linkinds(ψ)
    s1, s2 = siteinds(ψ)

    l02n = Index(dim(l02), "Link,n=0,l=2")
    l11n = Index(dim(l11), "Link,n=1,l=1")
    l12n = Index(dim(l12), "Link,n=1,l=2")
    l21n = Index(dim(l21), "Link,n=2,l=1")
    l22n = Index(dim(l22), "Link,n=2,l=2")
    s1n  = Index(dim(s1),  "Site,n=1")
    s2n  = Index(dim(s2),  "Site,n=2")

    Γa = setprime(replaceinds(ψ.Γa, (l02=>l02n, s1=>s1n, l11=>l11n)), 0)
    λa = setprime(replaceinds(ψ.λa, (l11=>l11n, l12=>l12n)), 0)
    Γb = setprime(replaceinds(ψ.Γb, (l12=>l12n, s2=>s2n, l21=>l21n)), 0)
    λb = setprime(replaceinds(ψ.λb, (l21=>l21n, l22=>l22n)), 0)
    return iMPS2(Γa, λa, Γb, λb)
end


maxbonddim(ψ::iMPS2) = maximum([dim(ind) for ind in linkinds(ψ)])

maxbonddim (generic function with 1 method)

In [ ]:
function eigenvec(A::ITensor, in_leg::Index{Int}, out_leg::Index{Int}; tol=1e-8, eager=true, krylovdim=40)
    Adag = dag(prime(A; tags="Link"))
    # println(inds(Adag))
    W = delta(in_leg, out_leg) * delta(prime(in_leg), prime(out_leg))
    
    map(X::ITensor) = W * (A * (Adag * X))

    v0 = randomITensor(in_leg, prime(in_leg))

    arn = Arnoldi(; tol, eager, krylovdim)

    TT, v, μ, info = schursolve(map, v0, 1, :LM, arn)
    μ = μ[1]
    V = v[1]

    if info.converged == 0
        @warn "map not converged after $(info.numiter) iterations"
    end
    if size(TT, 2) > 1 && TT[2, 1] != 0
        @warn "Non-unique largest eigenvector of map found"
    end

    return V, μ
end

evolve_AB(ψ::iMPS2, M::Matrix; kwargs...) = evolve(ψ, M; kwargs...)
evolve_BA(ψ::iMPS2, M::Matrix; kwargs...) = shift_cell(evolve(shift_cell(ψ), M; kwargs...))

function evolve(ψ::iMPS2, M::Matrix; maxdim=128, cutoff=1e-8, tol=1e-8, eager=true, krylovdim=40)
    Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb
    l02, l11, l12, l21, l22 = linkinds(ψ)
    s1, s2 = siteinds(ψ)
    l01 = Index(dim(l02), tags="Link,n=0,l=1")

    gate = op(M, [s1, s2])
    Φ = (Γa * λa * Γb) * gate
    Φ = replaceinds(Φ, prime(s1) => s1, prime(s2) => s2)

    Θ1 = Φ * λb
    VR, η = eigenvec(Θ1, l22, l02; tol=tol, eager=eager, krylovdim=krylovdim)

    Θ2 = replaceinds(λb, l21 => l01, l22 => l02) * Φ
    VL, τ = eigenvec(Θ2, l01, l21; tol=tol, eager=eager, krylovdim=krylovdim)
    
    Θ = Θ2 * λb

    X = decompose(VR, l22, prime(l22))
    Y = decompose(VL, prime(l01), l01) # actually Y this time (not Y^T)

    # are these index reuses safe? (next 4 lines)
    U, λb_new, V, _, u, v = svd(ITensor(transpose(Y), prime(l01, 2), l01) * ITensor(X, l01, prime(l01)), prime(l01, 2); maxdim=maxdim, cutoff=cutoff)
    λb_new /= norm(λb_new)

    λBVXinv = λb_new * V * ITensor(inv(X), prime(l01), l01)
    YTinvUλB = ITensor(inv(transpose(Y)), l22, prime(l22)) * replaceinds(U, prime(l01, 2) => prime(l22)) * λb_new

    Σ = λBVXinv * Θ * YTinvUλB
    # return Σ

    P, λa_new, Q, _, a, b = svd(Σ, u, s1; maxdim=maxdim, cutoff=cutoff)
    λa_new /= norm(λa_new)

    λb_inv_mat = diagm(diag(Matrix(λb_new, u, v).^(-1)))

    Γa_new = ITensor(λb_inv_mat, prime(u), u) * P
    Γb_new = Q * ITensor(λb_inv_mat, v, prime(v))
    λb_new = replaceinds(λb_new, v => prime(v))

    # println(Γa_new)
    # println(Γb_new)
    # println(λa_new)
    # println(λb_new)

    # Γa_new = ITensor(λb_inv_mat, l02, u) * replaceinds(P, a => l11)
    # Γb_new = replaceinds(Q, b => l12) * ITensor(λb_inv_mat, v, l21)
    # λa_new = replaceinds(λa_new, a => l11, b => l12)
    # λb_new = replaceinds(λb_new, u => l21, v => l22)
    return reset_indices(iMPS2(Γa_new, λa_new, Γb_new, λb_new))
end

function decompose(M::ITensor, in_leg::Index{Int}, out_leg::Index{Int})
    U, S, V, _, u, v = svd(M, in_leg)

    X = Matrix(U, in_leg, u) * Diagonal(sqrt.(diag(Matrix(S, u, v))))

    return X
end

function shift_cell(ψ::iMPS2)
    Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb

    l02, l11, l12, l21, l22 = linkinds(ψ)
    s1, s2 = siteinds(ψ) 

    l02_new = replacetags(l12, tags(l12) => tags(l02))
    l11_new = replacetags(l21, tags(l21) => tags(l11))
    l12_new = replacetags(l22, tags(l22) => tags(l12))
    l21_new = replacetags(l11, tags(l11) => tags(l21))
    l22_new = replacetags(l12, tags(l12) => tags(l22))

    s1_new = replacetags(s2, tags(s2) => tags(s1))
    s2_new = replacetags(s1, tags(s1) => tags(s2))

    Γa_new = replaceinds(Γb, l12 => l02_new, s2 => s1_new, l21 => l11_new)
    λa_new = replaceinds(λb, l21 => l11_new, l22 => l12_new)
    Γb_new = replaceinds(Γa, l02 => l12_new, s1 => s2_new, l11 => l21_new)
    λb_new = replaceinds(λa, l11 => l21_new, l12 => l22_new)

    shifted_ψ = iMPS2(Γa_new, λa_new, Γb_new, λb_new)

    return shifted_ψ
end

function linkinds(ψ::iMPS2)
    Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb

    l02 = only(setdiff(inds(Γa; tags="Link"), inds(λa)))
    l11 = commonind(Γa, λa)
    l12 = commonind(λa, Γb)
    l21 = commonind(Γb, λb)
    l22 = only(setdiff(inds(λb), inds(Γb)))

    return l02, l11, l12, l21, l22
end

function siteinds(ψ::iMPS2)
    Γa, λa, Γb, λb = ψ.Γa, ψ.λa, ψ.Γb, ψ.λb

    s1 = only(inds(Γa; tags="Site"))
    s2 = only(inds(Γb; tags="Site"))

    return s1, s2
end

siteinds (generic function with 1 method)

In [12]:

function dephasing_gate(M::Matrix, p::Float64)
    return (1-p) * I + p * M
end


SWAP = [1 0 0 0
        0 0 1 0
        0 1 0 0
        0 0 0 1]



p = 0.1

for _ in 1:100
    ψ = evolve_AB(ψ, dephasing_gate(SWAP, p); cutoff=1e-8, tol=1e-8)
    println("Max dim: ", maxbonddim(ψ))
    ψ = evolve_BA(ψ, dephasing_gate(SWAP, p); cutoff=1e-8, tol=1e-8)
    println("Max dim: ", maxbonddim(ψ))
end

((dim=1|id=313|"Link,l=2,n=0")', (dim=2|id=286|"Site,n=1"), (dim=2|id=153|"Site,n=2"), (dim=1|id=313|"Link,l=2,n=2")')
((dim=1|id=932|"Link,l=1,n=0")', (dim=1|id=311|"Link,l=1,n=2")', (dim=2|id=286|"Site,n=1"), (dim=2|id=153|"Site,n=2"))
Max dim: 1
((dim=1|id=357|"Link,l=2,n=0")', (dim=2|id=184|"Site,n=1"), (dim=2|id=727|"Site,n=2"), (dim=1|id=357|"Link,l=2,n=2")')
((dim=1|id=370|"Link,l=1,n=0")', (dim=1|id=670|"Link,l=1,n=2")', (dim=2|id=184|"Site,n=1"), (dim=2|id=727|"Site,n=2"))
Max dim: 1
((dim=1|id=69|"Link,l=2,n=0")', (dim=2|id=83|"Site,n=1"), (dim=2|id=86|"Site,n=2"), (dim=1|id=69|"Link,l=2,n=2")')
((dim=1|id=563|"Link,l=1,n=0")', (dim=1|id=83|"Link,l=1,n=2")', (dim=2|id=83|"Site,n=1"), (dim=2|id=86|"Site,n=2"))
Max dim: 1
((dim=1|id=564|"Link,l=2,n=0")', (dim=2|id=512|"Site,n=1"), (dim=2|id=489|"Site,n=2"), (dim=1|id=564|"Link,l=2,n=2")')
((dim=1|id=919|"Link,l=1,n=0")', (dim=1|id=628|"Link,l=1,n=2")', (dim=2|id=512|"Site,n=1"), (dim=2|id=489|"Site,n=2"))
Max dim: 1
((dim=1|id=

In [16]:
Array(ψ.Γa * ψ.λa * ψ.Γb * ψ.λb, inds(ψ.Γa * ψ.λa * ψ.Γb * ψ.λb)...)[:,1,:,1]

2×2 Matrix{ComplexF64}:
 -0.500002+7.69807e-5im         0.5-1.92392e-13im
       0.5-1.92392e-13im  -0.499998-7.69801e-5im